**Dataset Loading and Inspection**

In [13]:
import pandas as pd

iris_data = pd.read_csv('drive/MyDrive/Datasets For ML/Iris.csv')

print("Columns:", iris_data.columns)
print("\nShape:", iris_data.shape)
print("\nDuplicated:", iris_data.duplicated().sum())
print("\nNull Values:", iris_data.isnull().sum().sum())
print("\nFirst 2 rows:")
iris_data.head(2)

Columns: Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species'],
      dtype='object')

Shape: (150, 6)

Duplicated: 0

Null Values: 0

First 2 rows:


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


**Data Cleaning:** Drop unnecessary Id column. Encode target labels.

In [14]:
iris_data = iris_data.drop('Id', axis=1)
iris_data.shape

(150, 5)

In [15]:
from sklearn.preprocessing import LabelEncoder

print("Labels Before Encoding: ", iris_data.Species.unique())

le = LabelEncoder()
iris_data['Species'] = le.fit_transform(iris_data['Species'])

print("\nLabels After Encoding: ", iris_data.Species.unique())

Labels Before Encoding:  ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']

Labels After Encoding:  [0 1 2]


**Feature Scaling and Train-Test Split**

**Why Scaling?**

KNN relies on distance. Unscaled features distort results.

In [16]:
from sklearn.preprocessing import MinMaxScaler

X, Y = iris_data.iloc[:, :-1], iris_data.iloc[:, -1]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head(2)

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,0.222222,0.625000,0.067797,0.041667
1,0.166667,0.416667,0.067797,0.041667


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

**Radius-Based KNN (Unweighted)**

Classifies using all neighbors within a fixed radius

No fixed k

Sensitive to feature scale

**Radius KNN (Uniform Weights)**

In [18]:
from sklearn.neighbors import RadiusNeighborsClassifier

acc1 = []

for r in range(1, 6):
  knn = RadiusNeighborsClassifier(radius=r)
  knn.fit(X_train, Y_train)
  acc = knn.score(X_test, Y_test)
  acc1.append(acc)
  print(f"Radius: {r}, Accuracy: {acc}")

Radius: 1, Accuracy: 0.5333333333333333
Radius: 2, Accuracy: 0.3
Radius: 3, Accuracy: 0.3
Radius: 4, Accuracy: 0.3
Radius: 5, Accuracy: 0.3


**Radius-Based KNN (Distance Weighted)**

**Why Distance Weighting?**

Closer points influence prediction more

Reduces noise impact

In [19]:
acc2 = []

for r in range(1, 6):
    knn = RadiusNeighborsClassifier(radius=r, weights='distance', metric='euclidean')
    knn.fit(X_train, Y_train)
    acc = knn.score(X_test, Y_test)
    acc2.append(acc)
    print(f"Radius: {r}, Accuracy: {acc}")

Radius: 1, Accuracy: 1.0
Radius: 2, Accuracy: 1.0
Radius: 3, Accuracy: 1.0
Radius: 4, Accuracy: 1.0
Radius: 5, Accuracy: 1.0


**Optimal Radius Selection (Simple Validation Logic)**

**Selection Strategy**

Compare best accuracy from both configurations

Choose higher-performing radius

In [20]:
if max(acc1) >= max(acc2):
    optimal_radius = acc1.index(max(acc1)) + 1
    neigh = RadiusNeighborsClassifier(radius=optimal_radius)
else:
    optimal_radius = acc2.index(max(acc2)) + 1
    neigh = RadiusNeighborsClassifier(radius=optimal_radius, weights='distance', metric='euclidean')

print("Optimal Radius:", optimal_radius)
neigh.fit(X_train, Y_train)

Optimal Radius: 1


RadiusNeighborsClassifier(metric='euclidean', radius=1, weights='distance')

**Prediction on Test Data**

In [21]:
Y_pred = neigh.predict(X_test)
print("Predicted Labels:", Y_pred)

Predicted Labels: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


**Feature Range Analysis**

**Why?**

Helps choose reasonable radius values

Explains model behavior

In [22]:
print("Minimum values:\n", X.min())
print("\nMaximum values:\n", X.max())

Minimum values:
 SepalLengthCm    4.3
SepalWidthCm     2.0
PetalLengthCm    1.0
PetalWidthCm     0.1
dtype: float64

Maximum values:
 SepalLengthCm    7.9
SepalWidthCm     4.4
PetalLengthCm    6.9
PetalWidthCm     2.5
dtype: float64


**Prediction for New Sample**

In [23]:
sample_data = pd.DataFrame(
    [[4.4, 2.4, 1.9, 3.5]],
    columns=X.columns
)

sample_scaled = pd.DataFrame(scaler.transform(sample_data), columns=X.columns)

neighbors = neigh.radius_neighbors(sample_scaled)
print("Nearest neighbors:", neighbors)

print(f"\nPredicted Class: {neigh.predict(sample_scaled)} or {le.inverse_transform(neigh.predict(sample_scaled))}")

Nearest neighbors: (array([array([0.88188622, 0.89377427, 0.8877276 , 0.82658946, 0.97238996,
              0.95156311, 0.95156311])                                   ],
      dtype=object), array([array([116,  40, 112,  56,  89,  10,  43])], dtype=object))

Predicted Class: [2] or ['Iris-virginica']


**Conclusion**

Radius-based KNN classifies data using all points within a fixed radius. Feature scaling is important for correct distance computation. Distance weighting improves accuracy by giving closer points more influence. Optimal radius selection ensures the best performance. Predictions on test and new samples show that the model can classify accurately when radius and weighting are properly chosen.
